# Figure 5 — Predicted vs. actual Depth to Water under the four RIFT validation strategies

This notebook runs **one representative model with fixed hyperparameters** through the four data-splitting
strategies of the RIFT framework and plots the **test-set** predicted-vs-actual scatter for each:

- **(A) Random split** — 70/30 random (idealised interpolation benchmark).
- **(B) Well-based split** — entire wells held out (prediction at unmonitored locations).
- **(C) Spatial-grid split** — 10×10 geographic blocks held out (extrapolation to new regions).
- **(D) Temporal split** — last 30 % of each well's record held out (forecasting).

Using a *single fixed pipeline* across all four panels is deliberate: it isolates the effect of the
train–test dependence structure (the manuscript's central point), rather than comparing different fitted models.
Each panel shows the 1:1 line and the test-set R², RMSE, MAPE and sample size.

> ⚠️ This notebook **trains models**, so it is heavier than the figure notebooks. With the default
> Extra Trees (100 trees) on ~10⁵ rows it typically runs in a few minutes on a laptop.


In [ ]:
# === Configuration =========================================================
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
warnings.filterwarnings("ignore")

DATA_PATH  = r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE.xlsx"
OUTPUT_DIR = r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\draft_for_AI_model\figures_7_26"

# --- Key columns ---
DTW_COL   = "Depth to water (m)"           # target
LON_COL   = "Longitude_GCS_WGS_1984"
LAT_COL   = "Latitude_GCS_WGS_1984"
WELL_COL  = "Name"                          # well identifier (falls back to "ID")
DATE_COL  = "Date_String"                               # observation date for the temporal split;
                                             # "" -> auto-detect (Date_String / Date / Fecha ...)

# --- Representative model (same hyperparameters used for all four splits) ---
MODEL_NAME = "Extra Trees"                   # "Extra Trees" | "Random Forest" | "Decision Tree"
MODELS = {
    "Extra Trees":   ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
}

RANDOM_STATE = 42
TEST_MODE = False                            # True -> structured synthetic data (no Excel needed)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Model:", MODEL_NAME, "| output:", OUTPUT_DIR)


In [ ]:
# === Load data =============================================================
if TEST_MODE:
    rng = np.random.default_rng(0)
    n_wells = 600
    rows = []
    for w in range(n_wells):
        la = rng.uniform(-42, -18)
        lo = -71 + 0.05 * (la + 30) + rng.normal(0, 0.6)
        elev = max(0, (abs(la) - 18) * 90 + rng.normal(0, 200))
        offset = rng.normal(0, 8)                       # site-specific level
        trend = rng.uniform(0.0, 0.6)                   # depletion per year
        nrec = int(rng.integers(5, 180))
        years = np.sort(rng.uniform(1990, 2021, nrec))
        base = 6 + 0.25 * elev / 50 + 0.3 * (abs(la) - 18)
        dtw = base + offset + trend * (years - 2005) + rng.normal(0, 2, nrec)
        for yr, d in zip(years, dtw):
            rows.append((f"W{w:04d}", lo + rng.normal(0, 0.01), la + rng.normal(0, 0.01),
                         elev, max(0.05, d), pd.Timestamp("2000-01-01") + pd.to_timedelta((yr-2000)*365.25, "D"),
                         f"Basin_{int((la+18)//2):02d}", "Active"))
    df = pd.DataFrame(rows, columns=[WELL_COL, LON_COL, LAT_COL, "elevation_m", DTW_COL,
                                     "Date_String", "Basin", "Status"])
    # a couple of satellite-like predictors correlated with location
    df["terraclim_pr_value"] = 200 - 3 * (abs(df[LAT_COL]) - 18) + rng.normal(0, 20, len(df))
    df["wclim_bio_bio02_value"] = 8 + 0.2 * (abs(df[LAT_COL]) - 18) + rng.normal(0, 1, len(df))
    DATE_COL = "Date_String"
    print("TEST_MODE: synthetic dataset", df.shape)
else:
    df = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith((".xlsx", ".xls")) else pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}\nShape: {df.shape}")

# Resolve the well identifier and the date column
if WELL_COL not in df.columns:
    WELL_COL = "ID" if "ID" in df.columns else None
    if WELL_COL is None:
        raise KeyError("No well identifier column found (looked for 'Name' and 'ID'). Set WELL_COL.")
if not DATE_COL:
    for cand in ["Date_String", "Date", "date", "Fecha", "observation_date", "DATE"]:
        if cand in df.columns:
            DATE_COL = cand; break
if not DATE_COL or DATE_COL not in df.columns:
    raise KeyError("No date column found for the temporal split. Set DATE_COL to your date field.\n"
                   "Available columns:\n  - " + "\n  - ".join(map(str, df.columns)))
print("Well column:", WELL_COL, "| date column:", DATE_COL)


In [ ]:
# === Clean + build feature matrix (mirrors the analysis pipeline) ==========
df = df.dropna(subset=[DTW_COL, LON_COL, LAT_COL]).copy()
n_neg = int((df[DTW_COL] < 0).sum())
df = df[df[DTW_COL] >= 0].copy()
df["_date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=["_date"]).reset_index(drop=True)
print(f"Records after cleaning: {len(df):,}  (removed {n_neg:,} negative-depth)")

y = df[DTW_COL].astype(float).values

# Feature columns: all numeric + label-encoded categoricals, excluding the target.
# (Encoders are fit on the full dataset so identifier codes are consistent across splits,
#  reproducing the 'memorisation' behaviour discussed in the paper.)
exclude = {DTW_COL, "_date"}
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
cat_cols = [c for c in df.select_dtypes(include=["object"]).columns if c not in exclude]
for c in cat_cols:
    df[c + "_enc"] = LabelEncoder().fit_transform(df[c].astype(str))
feat_cols = num_cols + [c + "_enc" for c in cat_cols]
X = df[feat_cols].copy()
print(f"Features: {len(feat_cols)}  ({len(num_cols)} numeric + {len(cat_cols)} encoded categorical)")


In [ ]:
# === Define the four validation splits (return boolean test masks) =========
N = len(df)
idx = np.arange(N)
rs = np.random.RandomState(RANDOM_STATE)

# (A) Random 70/30
def split_random():
    test = np.zeros(N, bool)
    test[rs.choice(idx, size=int(0.30 * N), replace=False)] = True
    return test

# (B) Well-based: ~30% of wells entirely to test
def split_well():
    wells = df[WELL_COL].unique()
    test_wells = set(rs.choice(wells, size=int(0.30 * len(wells)), replace=False))
    return df[WELL_COL].isin(test_wells).values

# (C) Spatial grid: 10x10 lon/lat blocks, ~30% of (non-empty) cells to test
def split_grid(nx=10, ny=10):
    gx = pd.cut(df[LON_COL], bins=nx, labels=False)
    gy = pd.cut(df[LAT_COL], bins=ny, labels=False)
    cell = gx.astype(str) + "_" + gy.astype(str)
    cells = cell.unique()
    test_cells = set(rs.choice(cells, size=max(1, int(0.30 * len(cells))), replace=False))
    return cell.isin(test_cells).values

# (D) Temporal: last 30% of each well's chronological record (wells with >3 records)
def split_temporal():
    test = np.zeros(N, bool)
    order = df.sort_values([WELL_COL, "_date"]).index
    for _, g in df.loc[order].groupby(WELL_COL, sort=False):
        m = len(g)
        if m > 3:
            k = int(np.ceil(0.30 * m))
            test[g.index[-k:]] = True   # most recent k records -> test
    return test

SPLITS = {"(A) Random split": split_random,
          "(B) Well-based split": split_well,
          "(C) Spatial-grid split": split_grid,
          "(D) Temporal split": split_temporal}


In [ ]:
# === Train the fixed model under each split, collect test predictions ======
def evaluate(test_mask):
    tr, te = ~test_mask, test_mask
    imp = SimpleImputer(strategy="median").fit(X[tr])          # fit on TRAIN only
    Xtr = imp.transform(X[tr]); Xte = imp.transform(X[te])
    model = MODELS[MODEL_NAME]
    model.fit(Xtr, y[tr])
    pred = model.predict(Xte)
    yte = y[te]
    r2 = r2_score(yte, pred)
    rmse = float(np.sqrt(mean_squared_error(yte, pred)))
    pos = yte > 0
    mape = float(mean_absolute_percentage_error(yte[pos], pred[pos]) * 100)
    return yte, pred, dict(r2=r2, rmse=rmse, mape=mape, n=int(te.sum()))

results = {}
for name, fn in SPLITS.items():
    mask = fn()
    results[name] = evaluate(mask)
    m = results[name][2]
    print(f"{name:<24} R2={m['r2']:6.3f}  RMSE={m['rmse']:6.2f} m  MAPE={m['mape']:7.1f}%  n_test={m['n']:,}")


In [ ]:
# === Build Figure 5 (2x2 predicted vs actual) ==============================
plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.titlesize": 10, "axes.linewidth": 0.6})

# common axis limit so panels are directly comparable
hi = float(np.percentile(np.concatenate([r[0] for r in results.values()]), 99.5))
hi = np.ceil(hi / 10) * 10
LIM = (0, hi)

fig, axes = plt.subplots(2, 2, figsize=(9.2, 9.4))
axes = axes.ravel()
ORANGE = "#E1812C"

for ax, (name, (yte, pred, m)) in zip(axes, results.items()):
    ax.scatter(yte, pred, s=5, alpha=0.25, color=ORANGE, edgecolor="none",
               rasterized=True, zorder=2)
    ax.plot(LIM, LIM, "k--", lw=1.1, zorder=3)              # 1:1 line
    ax.set_xlim(*LIM); ax.set_ylim(*LIM); ax.set_aspect("equal")
    ax.set_title(name, fontweight="bold", loc="left")
    ax.set_xlabel("Actual DTW (m)"); ax.set_ylabel("Predicted DTW (m)")
    ax.grid(True, ls=":", lw=0.4, color="0.85")
    txt = (f"$R^2$ = {m['r2']:.3f}\n"
           f"RMSE = {m['rmse']:.2f} m\n"
           f"MAPE = {m['mape']:.1f}%\n"
           f"n = {m['n']:,}")
    ax.text(0.04, 0.96, txt, transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.7", lw=0.6), zorder=4)

fig.suptitle(f"Figure 5. Predicted vs. actual Depth to Water across validation strategies "
             f"({MODEL_NAME}, test set)", fontsize=11, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.975])
plt.show()


In [ ]:
# === Save (PNG 400 dpi + PDF + SVG) ========================================
stem = os.path.join(OUTPUT_DIR, "Figure5_pred_vs_actual_strategies")
fig.savefig(stem + ".png", dpi=400, bbox_inches="tight")
fig.savefig(stem + ".pdf", bbox_inches="tight")
fig.savefig(stem + ".svg", bbox_inches="tight")
print("Saved:")
for ext in (".png", ".pdf", ".svg"):
    print("  ", stem + ext)
